# 02 Data Exploration

This notebook profiles the extracted MIMIC-III tables needed for multimodal sepsis research. It is designed to be chunk-aware, Colab-friendly, and reproducible so later notebooks can rely on saved exploration artifacts instead of hidden in-memory state.

## Objectives

- Validate that the extracted tables are readable.
- Build schema previews for the key research tables.
- Estimate table sizes and cohort identifiers.
- Summarize missingness from a configurable sample.
- Profile note category coverage for the clinical text pipeline.
- Save reusable outputs for downstream notebooks and the paper.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/multimodal-early-sepsis') if IN_COLAB else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
if IN_COLAB:
    %pip -q install pyyaml pandas

In [ ]:
import pandas as pd

from src.utils.config import load_config
from src.utils.paths import ensure_directories, resolve_project_paths
from src.utils.logging_utils import write_run_manifest
from src.utils.io_utils import save_dataframe_bundle
from src.data_processing.io import list_available_tables, validate_required_tables
from src.data_processing.exploration import build_exploration_bundle, summarize_demographics

In [ ]:
base_config_path = PROJECT_ROOT / 'configs' / 'base.yaml'
override_config_path = PROJECT_ROOT / 'configs' / 'colab.yaml' if IN_COLAB else None
config = load_config(base_config_path, override_config_path)
paths = resolve_project_paths(config)
ensure_directories(paths)

available_tables = list_available_tables(paths['extracted_data_dir'])
required_status = validate_required_tables(paths['extracted_data_dir'], config['dataset']['required_tables'])
available_tables[:20], required_status

In [ ]:
assert all(required_status.values()), 'Some required tables are missing. Run 01_dataset_setup first.'

table_names = [
    table_name for table_name in config['exploration']['key_tables']
    if table_name in available_tables
]

table_names

## Build exploration artifacts

The profiling functions below read only the amount of data they need. Large tables use chunked row counting, while schema and missingness summaries use small previews or samples.

In [ ]:
bundle = build_exploration_bundle(
    extracted_dir=paths['extracted_data_dir'],
    table_names=table_names,
    preview_rows=config['exploration']['preview_rows'],
    sample_rows=config['exploration']['missingness_sample_rows'],
    chunksize=config['dataset']['chunksize'],
    low_memory=config['dataset']['low_memory'],
    id_columns=config['exploration']['cohort_id_columns'],
    note_category_top_k=config['exploration']['note_category_top_k'],
)
demographics_summary = summarize_demographics(
    extracted_dir=paths['extracted_data_dir'],
    low_memory=config['dataset']['low_memory'],
)
bundle.keys(), demographics_summary

In [ ]:
artifact_dir = paths['tables_dir'] / '02_data_exploration'
saved_paths = save_dataframe_bundle(bundle, artifact_dir)
saved_paths

## Preview saved outputs

In [ ]:
bundle['table_summary']

In [ ]:
bundle['schema_preview'].head(20)

In [ ]:
bundle['missingness_sample'].sort_values(['table_name', 'missing_fraction_sample'], ascending=[True, False]).head(30)

In [ ]:
bundle['note_category_summary']

In [ ]:
pd.DataFrame([
    {'metric': key, 'value': value}
    for key, value in demographics_summary.items()
])

In [ ]:
manifest_path = paths['manifests_dir'] / '02_data_exploration_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='02_data_exploration',
    config=config,
    extra={
        'table_names_profiled': table_names,
        'saved_artifacts': saved_paths,
        'demographics_summary': demographics_summary,
    },
)
manifest_path

## Research interpretation prompts

Use the generated tables to answer the following before cohort construction:

- Which tables dominate storage and runtime?
- How complete are note timestamps for temporal alignment?
- Which identifiers are available consistently across tables?
- Which variables are so sparse that they may need exclusion or special handling later?
- Are the selected note categories sufficiently represented for multimodal modeling?

## Next notebook

`03_cohort_construction.ipynb` will use these exploration outputs to build the ICU cohort, implement Sepsis-3 labeling, and define leakage-safe patient-level splits.